# 20 Clean the data: list coverage, imputation and drops

Notebook 10 (`10_load-data.ipynb`) looked at the meter data as a whole and exported the number of rows each customer has. This notebook focuses on three groups of customers that Avangrid has identified in `EV_Heat Pump_Solar PV Customers.xlsx`:

- **EV**: customers believed to have an electric vehicle.
- **HP**: customers believed to have a heat pump.
- **PV**: customers with solar panels.

These technologies change how and when a household uses electricity, so they are likely to matter for any analysis of the load data. Before using these groups, we check how complete their meter data is.

Run this notebook from the project folder, the one containing `onedrive-export/`, after notebook 10 has created `row_counts_by_customer.csv`.

## Setup

In [1]:
import duckdb
import polars as pl

## Load the row counts per customer

This file was produced in notebook 10. It holds one row per customer with the number of hourly rows they have in the meter data.

A complete year is **8,758 rows**: 365 days × 24 hours, minus one missing hour on each daylight-saving changeover day. Any other count means the customer's data is **incomplete**, so we flag that here once and reuse it below.

In [2]:
ROW_COUNTS_PATH = 'onedrive-export/Data/eda/row_counts_by_customer.csv'
EXPECTED_ROWS = 8_758

row_counts = pl.read_csv(ROW_COUNTS_PATH).with_columns(
    (pl.col('row_count') != EXPECTED_ROWS).alias('is_incomplete')
)
print(f'{row_counts.height:,} customers in the meter data, '
      f'{row_counts["is_incomplete"].sum():,} with incomplete data.')
row_counts.head()

34,238 customers in the meter data, 8,941 with incomplete data.


CUSTOMER_NUMBER,row_count,is_incomplete
i64,i64,bool
1,7044,true
2,8024,true
3,8758,false
4,8758,false
5,8566,true


## Load the EV, heat pump and solar PV customer lists

The workbook has one tab per customer group, each with a single `CUSTOMER_NUMBER` column, plus a **Description** tab that explains how each list was put together. `sheet_id=0` loads every tab at once into a dictionary keyed by tab name.

The Description tab is text, not a customer list, so we print it for context and keep it out of the counts. It is important for reading the results:

- the **EV** and **HP** lists are **random samples** (about 25% of each group), and membership is what Avangrid *believes*, not confirmed;
- the **PV** list is the **full** list of solar customers.

In [3]:
CUSTOMER_GROUPS_PATH = 'onedrive-export/Data/EV_Heat Pump_Solar PV Customers.xlsx'

sheets = pl.read_excel(CUSTOMER_GROUPS_PATH, sheet_id=0)
print('Tabs:', list(sheets))

description = sheets.pop('Description')
print()
for line in [description.columns[0], *description.to_series()]:
    print('-', line.strip())

Tabs: ['Description', 'EV Customers', 'HP Customers', 'PV Customers']

- The EV Customers sheet contains a random sampling of 200 customers that we believe may have an electric vehicle. This is about 25% of the electrici vehicle population in the full sample.
- The HP Customers sheet contains a random sampling of 100 customers that we believe may have a heat pump. This is about 25% of the heat pump populaton in the full sample.
- The PV Customers sheet includes a full list of all customers that have solar PV.


## Meter data by folder

The meter data was delivered in four folders, named by customer-number range. Avangrid first said the folder **`2026 Avangrid Hackathon Data_10001_20000`** held duplicate data and should be disregarded, then supplied a corrected version of it. **All four folders are now used, and no customer numbers are excluded.**

The corrected folder arrived incomplete: 7 of its 51 files (part 41 and parts 44–49) were missing, so most customers numbered 18,152–18,346 and 18,747–19,953 have no meter data yet. Listed customers among them count as having *no meter data* below, like any other listed customer without data. The counts per folder keep the gap visible; re-run from notebook 10 once the missing files arrive.

In [4]:
FOLDER_RANGES = [(1, 10_000), (10_001, 20_000), (20_001, 30_000), (30_001, 35_725)]

for lo, hi in FOLDER_RANGES:
    n = row_counts.filter(pl.col('CUSTOMER_NUMBER').is_between(lo, hi)).height
    print(f'{lo:,}–{hi:,}: {n:,} of {hi - lo + 1:,} customer numbers have meter data')

groups = dict(sheets)  # every listed customer counts; none are excluded

1–10,000: 9,984 of 10,000 customer numbers have meter data
10,001–20,000: 8,585 of 10,000 customer numbers have meter data
20,001–30,000: 9,965 of 10,000 customer numbers have meter data
30,001–35,725: 5,704 of 5,725 customer numbers have meter data


## Customers and incomplete data on each tab

For each tab we report:

- **customers**: customers on the tab. This is the **denominator**;
- **complete**: customers with exactly 8,758 rows;
- **incomplete**: customers without 8,758 rows. This **includes** customers with no meter data at all, because for deciding whether to drop customers, having no data is the most incomplete case;
- **no_meter_data**: the part of *incomplete* with no rows at all, shown separately so these customers aren't hidden;
- **share_incomplete**: incomplete ÷ customers.

A **left join** from each tab to the row counts keeps every listed customer, including those with no meter data.

In [5]:
group_counts = pl.concat([
    customers.join(row_counts, on='CUSTOMER_NUMBER', how='left')
    .with_columns(pl.lit(tab).alias('tab'))
    for tab, customers in groups.items()
])

summary = (
    group_counts.group_by('tab', maintain_order=True)
    .agg(
        pl.len().alias('customers'),
        (pl.col('row_count') == EXPECTED_ROWS).sum().alias('complete'),
        pl.col('is_incomplete').fill_null(True).sum().alias('incomplete'),
        pl.col('row_count').is_null().sum().alias('no_meter_data'),
    )
    .with_columns((pl.col('incomplete') / pl.col('customers')).round(3).alias('share_incomplete'))
)
summary

tab,customers,complete,incomplete,no_meter_data,share_incomplete
str,u32,u32,u32,u32,f64
"""EV Customers""",200,132,68,14,0.34
"""HP Customers""",100,54,46,12,0.46
"""PV Customers""",434,321,113,37,0.26


For comparison, this is the share of **all** customers in the meter data with incomplete data. A much higher share for one group means that dropping incomplete customers would cut that group harder than the rest.

In [6]:
print(f'All customers: {row_counts["is_incomplete"].mean():.1%} of '
      f'{row_counts.height:,} have incomplete data.')

All customers: 26.1% of 34,238 have incomplete data.


## How much data are incomplete customers missing?

Before dropping incomplete customers, it helps to know **how** incomplete they are. A customer missing one hour is very different from one missing six months.

We express each customer's gap as **missing hours** (8,758 − row count) and group the gaps into bands. Customers with no meter data are shown as their own band.

Notebook 10 found a few dates where many customers had no data at once, such as 2025-12-08, 2026-03-10 and 2026-06-03. These look like system-wide outages. A customer whose only gap is one of those days is missing about 24 hours through no fault of their own meter.

In [7]:
all_customers = row_counts.select('CUSTOMER_NUMBER', 'row_count').with_columns(pl.lit('All customers').alias('tab'))

with_gaps = pl.concat([
    group_counts.select('CUSTOMER_NUMBER', 'row_count', 'tab'),
    all_customers,
]).with_columns((EXPECTED_ROWS - pl.col('row_count')).alias('missing_hours'))

GAP_BANDS = ['0 (complete)', '1–24', '25–48', '49–168 (up to 1 week)',
             '169–720 (up to 30 days)', 'more than 720', 'no meter data']

gap_bands = (
    with_gaps.with_columns(
        pl.when(pl.col('missing_hours').is_null()).then(pl.lit(GAP_BANDS[6]))
        .when(pl.col('missing_hours') <= 0).then(pl.lit(GAP_BANDS[0]))
        .when(pl.col('missing_hours') <= 24).then(pl.lit(GAP_BANDS[1]))
        .when(pl.col('missing_hours') <= 48).then(pl.lit(GAP_BANDS[2]))
        .when(pl.col('missing_hours') <= 168).then(pl.lit(GAP_BANDS[3]))
        .when(pl.col('missing_hours') <= 720).then(pl.lit(GAP_BANDS[4]))
        .otherwise(pl.lit(GAP_BANDS[5]))
        .cast(pl.Enum(GAP_BANDS)).alias('missing_hours_band')
    )
    .pivot(on='tab', index='missing_hours_band', values='CUSTOMER_NUMBER',
           aggregate_function='len', sort_columns=False)
    .sort('missing_hours_band')
    .fill_null(0)
)
with pl.Config(tbl_rows=-1, tbl_hide_dataframe_shape=True, thousands_separator=True):
    print('Number of customers by missing hours (8,758 - row count):')
    print(gap_bands)

Number of customers by missing hours (8,758 - row count):
┌─────────────────────────┬──────────────┬──────────────┬──────────────┬───────────────┐
│ missing_hours_band      ┆ EV Customers ┆ HP Customers ┆ PV Customers ┆ All customers │
│ ---                     ┆ ---          ┆ ---          ┆ ---          ┆ ---           │
│ enum                    ┆ u32          ┆ u32          ┆ u32          ┆ u32           │
╞═════════════════════════╪══════════════╪══════════════╪══════════════╪═══════════════╡
│ 0 (complete)            ┆ 132          ┆ 54           ┆ 321          ┆ 25,297        │
│ 1–24                    ┆ 33           ┆ 20           ┆ 45           ┆ 5,205         │
│ 25–48                   ┆ 12           ┆ 8            ┆ 17           ┆ 2,555         │
│ 49–168 (up to 1 week)   ┆ 4            ┆ 4            ┆ 7            ┆ 564           │
│ 169–720 (up to 30 days) ┆ 0            ┆ 0            ┆ 4            ┆ 110           │
│ more than 720           ┆ 5            ┆ 2        

## How many customers would be kept at different cut-offs?

Dropping *every* incomplete customer is the strictest option. A looser rule keeps customers missing only a small amount of data, which in practice would mostly mean the outage days. Their few missing hours can be left out of averages, or filled in, without much effect.

The table shows the **share of customers kept** in each group if we keep everyone missing **at most** the given number of hours. Customers with no meter data are always dropped. Things to look for:

- how much each group shrinks under the strict rule (0 hours) compared with a looser one;
- whether the EV and HP groups, which are already small samples, shrink much more than the overall population. That would leave fewer customers to study and could bias the comparison.

In [8]:
MAX_MISSING_HOURS = [0, 24, 48, 168, 720]

kept = (
    with_gaps.group_by('tab', maintain_order=True)
    .agg(
        pl.len().alias('customers'),
        *[(pl.col('missing_hours') <= h).sum().alias(f'kept_max_{h}h') for h in MAX_MISSING_HOURS],
    )
)
kept_share = kept.select(
    'tab', 'customers',
    *[(pl.col(f'kept_max_{h}h') / pl.col('customers')).round(3).alias(f'share_kept_max_{h}h')
      for h in MAX_MISSING_HOURS],
)

with pl.Config(tbl_cols=-1, tbl_hide_dataframe_shape=True, thousands_separator=True):
    print('Customers kept:')
    print(kept)
    print('\nShare of customers kept:')
    print(kept_share)

Customers kept:
┌──────────────┬───────────┬─────────────┬──────────────┬──────────────┬─────────────┬─────────────┐
│ tab          ┆ customers ┆ kept_max_0h ┆ kept_max_24h ┆ kept_max_48h ┆ kept_max_16 ┆ kept_max_72 │
│ ---          ┆ ---       ┆ ---         ┆ ---          ┆ ---          ┆ 8h          ┆ 0h          │
│ str          ┆ u32       ┆ u32         ┆ u32          ┆ u32          ┆ ---         ┆ ---         │
│              ┆           ┆             ┆              ┆              ┆ u32         ┆ u32         │
╞══════════════╪═══════════╪═════════════╪══════════════╪══════════════╪═════════════╪═════════════╡
│ EV Customers ┆ 200       ┆ 132         ┆ 165          ┆ 177          ┆ 181         ┆ 181         │
│ HP Customers ┆ 100       ┆ 54          ┆ 74           ┆ 82           ┆ 86          ┆ 86          │
│ PV Customers ┆ 434       ┆ 321         ┆ 366          ┆ 383          ┆ 390         ┆ 394         │
│ All          ┆ 34,238    ┆ 25,297      ┆ 30,502       ┆ 33,057       ┆ 33

## Impute short gaps (up to three days)

The gap analysis showed that most incomplete customers are missing only **one to three whole days**, usually days when thousands of customers lost data at once. Their meters weren't faulty, so instead of dropping these customers we **fill in (impute)** the missing hours with a typical value for that customer.

**Which customers are imputed.** Customers missing **between 1 and 72 hours** in total (up to three days' worth). Customers missing more than that, such as those who joined late, left early or have long mid-year gaps, are not imputed. Filling weeks or months of data would mostly be invention.

**How each missing hour is filled.** A missing hour is filled with the average of the **same customer's** readings:

- in the **same week** (Monday to Sunday);
- at the **same hour of the day** (`hour_ending`);
- on the **same type of day**: weekday or weekend.

For example, if a Wednesday is missing, its 12 AM value (`hour_ending` 1) is the average of 12 AM on the other weekdays of that week (Monday, Tuesday, Thursday and Friday). If a Saturday is missing, its 12 AM value is that week's Sunday 12 AM value, the only other weekend day. Using the same week keeps the season and weather close to the missing day. Separating weekdays from weekends and matching the hour preserves the customer's daily and weekly routine.

Averages use only **real readings**, never other imputed values.

**Fallback.** If the week has no real reading of the same day type at that hour, we use the same average over the **previous and next weeks** combined. This mainly affects the last day of the data, Monday 2026-08-31, which is the only day of its week.

**Where the results go.** Imputed rows are written to a separate table, `ami_imputed_rows`, in `avangrid.db`, so the original `ami` table stays untouched. A view, `ami_filled`, combines the two and adds an `is_imputed` flag, so imputed values can always be told apart or excluded. Imputed rows have `interval_count` = 0 and `is_complete_hour` = 0, because no meter readings went into them. `imputation_source` records whether the value came from the same week or adjacent weeks.

The expected set of hourly slots is taken from the customers with complete data, so the hour that doesn't exist when clocks go forward is not "filled in".

In [9]:
DB_PATH = 'onedrive-export/Data/avangrid.db'
MAX_IMPUTE_HOURS = 72  # up to three days

IMPUTE_SQL = f"""
-- Hourly slots a complete customer has (8,758 of them)
CREATE OR REPLACE TEMP TABLE customer_rows AS
SELECT CUSTOMER_NUMBER, COUNT(*) AS row_count, ANY_VALUE(interval_type) AS interval_type
FROM ami GROUP BY 1;

CREATE OR REPLACE TEMP TABLE expected_slots AS
SELECT DISTINCT a.usage_date, a.hour_ending, a.day_type,
       CAST(date_trunc('week', a.usage_date) AS DATE) AS week_start   -- Monday
FROM ami a JOIN customer_rows c USING (CUSTOMER_NUMBER)
WHERE c.row_count = {EXPECTED_ROWS};

-- Customers missing 1 to {MAX_IMPUTE_HOURS} hours
CREATE OR REPLACE TEMP TABLE impute_customers AS
SELECT CUSTOMER_NUMBER, interval_type
FROM customer_rows
WHERE {EXPECTED_ROWS} - row_count BETWEEN 1 AND {MAX_IMPUTE_HOURS};

-- Their missing slots
CREATE OR REPLACE TEMP TABLE missing_slots AS
SELECT i.CUSTOMER_NUMBER, i.interval_type, s.*
FROM impute_customers i CROSS JOIN expected_slots s
ANTI JOIN ami a ON a.CUSTOMER_NUMBER = i.CUSTOMER_NUMBER
               AND a.usage_date = s.usage_date AND a.hour_ending = s.hour_ending;

-- Real readings summed by customer, week, day type and hour
CREATE OR REPLACE TEMP TABLE weekly AS
SELECT a.CUSTOMER_NUMBER, CAST(date_trunc('week', a.usage_date) AS DATE) AS week_start,
       a.day_type, a.hour_ending, SUM(a.hourly_kwh) AS kwh_sum, COUNT(*) AS n
FROM ami a SEMI JOIN impute_customers i USING (CUSTOMER_NUMBER)
GROUP BY ALL;

-- Same-week average, falling back to the previous and next weeks combined
CREATE OR REPLACE TABLE ami_imputed_rows AS
SELECT m.CUSTOMER_NUMBER, m.usage_date, m.hour_ending, m.day_type,
       COALESCE(w.kwh_sum / w.n,
                (COALESCE(p.kwh_sum, 0) + COALESCE(nx.kwh_sum, 0))
                / NULLIF(COALESCE(p.n, 0) + COALESCE(nx.n, 0), 0)) AS hourly_kwh,
       CAST(0 AS TINYINT) AS interval_count,
       m.interval_type,
       CAST(0 AS TINYINT) AS is_complete_hour,
       CASE WHEN w.n IS NOT NULL THEN 'same week'
            WHEN COALESCE(p.n, 0) + COALESCE(nx.n, 0) > 0 THEN 'adjacent weeks'
       END AS imputation_source
FROM missing_slots m
LEFT JOIN weekly w  ON w.CUSTOMER_NUMBER = m.CUSTOMER_NUMBER AND w.week_start = m.week_start
                   AND w.day_type = m.day_type AND w.hour_ending = m.hour_ending
LEFT JOIN weekly p  ON p.CUSTOMER_NUMBER = m.CUSTOMER_NUMBER AND p.week_start = m.week_start - 7
                   AND p.day_type = m.day_type AND p.hour_ending = m.hour_ending
LEFT JOIN weekly nx ON nx.CUSTOMER_NUMBER = m.CUSTOMER_NUMBER AND nx.week_start = m.week_start + 7
                   AND nx.day_type = m.day_type AND nx.hour_ending = m.hour_ending
ORDER BY 1, 2, 3;

-- Original plus imputed rows, flagged
CREATE OR REPLACE VIEW ami_filled AS
SELECT *, FALSE AS is_imputed, CAST(NULL AS VARCHAR) AS imputation_source FROM ami
UNION ALL BY NAME
SELECT *, TRUE AS is_imputed FROM ami_imputed_rows;
"""

with duckdb.connect(DB_PATH) as con:
    con.execute(IMPUTE_SQL)
    imputation_summary = con.sql("""
        SELECT imputation_source,
               COUNT(*) AS imputed_hours,
               COUNT(DISTINCT CUSTOMER_NUMBER) AS customers,
               COUNT(*) FILTER (WHERE hourly_kwh IS NULL) AS still_missing
        FROM ami_imputed_rows GROUP BY ALL ORDER BY 2 DESC
    """).pl()
    imputed_per_customer = con.sql("""
        SELECT CUSTOMER_NUMBER, COUNT(*) AS imputed_rows
        FROM ami_imputed_rows WHERE hourly_kwh IS NOT NULL GROUP BY 1
    """).pl()

print(f'Imputed {imputation_summary["imputed_hours"].sum():,} hours for '
      f'{imputed_per_customer.height:,} customers.')
imputation_summary

Imputed 279,026 hours for 8,269 customers.


imputation_source,imputed_hours,customers,still_missing
str,i64,i64,i64
"""same week""",278939,8267,0
"""adjacent weeks""",87,4,0


### Spot check

To confirm the rule works as described, we recompute a few imputed values by hand directly from `ami`: the average of the same customer, same week, same day type and same hour. We pick one missing weekday hour and one missing weekend hour. The `manual_average` column should equal `hourly_kwh`.

In [10]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    spot_check = con.sql("""
        WITH picks AS (
            (SELECT * FROM ami_imputed_rows
             WHERE imputation_source = 'same week' AND day_type = 'Weekday' LIMIT 1)
            UNION ALL
            (SELECT * FROM ami_imputed_rows
             WHERE imputation_source = 'same week' AND day_type = 'Weekend' LIMIT 1)
        )
        SELECT p.CUSTOMER_NUMBER, p.usage_date, dayname(p.usage_date) AS weekday,
               p.hour_ending, p.hourly_kwh,
               (SELECT AVG(a.hourly_kwh) FROM ami a
                WHERE a.CUSTOMER_NUMBER = p.CUSTOMER_NUMBER
                  AND a.hour_ending = p.hour_ending
                  AND a.day_type = p.day_type
                  AND date_trunc('week', a.usage_date) = date_trunc('week', p.usage_date)
               ) AS manual_average
        FROM picks p
    """).pl()

assert (spot_check['hourly_kwh'] - spot_check['manual_average']).abs().max() < 1e-9
spot_check

CUSTOMER_NUMBER,usage_date,weekday,hour_ending,hourly_kwh,manual_average
i32,date,str,i8,f64,f64
20,2025-09-01,"""Monday""",1,0.6995,0.6995
35,2026-01-10,"""Saturday""",1,0.977,0.977


## EV, HP and PV customers after imputation

We repeat the tab-by-tab completeness report, now counting imputed hours. As before, customers with no meter data count as incomplete.

- **complete_before**: customers with exactly 8,758 real rows;
- **complete_after**: customers with 8,758 rows once imputed hours are included;
- **still_incomplete**: customers whose gaps were too long to impute (more than 72 hours) or who have no meter data.

In [11]:
row_counts_filled = (
    row_counts.join(imputed_per_customer, on='CUSTOMER_NUMBER', how='left')
    .with_columns(
        (pl.col('row_count') + pl.col('imputed_rows').fill_null(0)).alias('row_count_filled')
    )
)

def completeness(customers: pl.DataFrame, tab: str) -> pl.DataFrame:
    joined = customers.join(row_counts_filled, on='CUSTOMER_NUMBER', how='left')
    return joined.select(
        pl.lit(tab).alias('tab'),
        pl.len().alias('customers'),
        (pl.col('row_count') == EXPECTED_ROWS).sum().alias('complete_before'),
        (pl.col('row_count_filled') == EXPECTED_ROWS).sum().alias('complete_after'),
        (pl.col('row_count_filled') != EXPECTED_ROWS).fill_null(True).sum().alias('still_incomplete'),
        pl.col('row_count').is_null().sum().alias('no_meter_data'),
    )

summary_after = pl.concat(
    [completeness(customers, tab) for tab, customers in groups.items()]
    + [completeness(row_counts.select('CUSTOMER_NUMBER'), 'All customers')]
).with_columns(
    (pl.col('complete_before') / pl.col('customers')).round(3).alias('share_complete_before'),
    (pl.col('complete_after') / pl.col('customers')).round(3).alias('share_complete_after'),
)

with pl.Config(tbl_cols=-1, tbl_hide_dataframe_shape=True, thousands_separator=True):
    print(summary_after)

┌────────────┬───────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┐
│ tab        ┆ customers ┆ complete_b ┆ complete_a ┆ still_inc ┆ no_meter_ ┆ share_com ┆ share_com │
│ ---        ┆ ---       ┆ efore      ┆ fter       ┆ omplete   ┆ data      ┆ plete_bef ┆ plete_aft │
│ str        ┆ u32       ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ore       ┆ er        │
│            ┆           ┆ u32        ┆ u32        ┆ u32       ┆ u32       ┆ ---       ┆ ---       │
│            ┆           ┆            ┆            ┆           ┆           ┆ f64       ┆ f64       │
╞════════════╪═══════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ EV         ┆ 200       ┆ 132        ┆ 179        ┆ 21        ┆ 14        ┆ 0.66      ┆ 0.895     │
│ Customers  ┆           ┆            ┆            ┆           ┆           ┆           ┆           │
│ HP         ┆ 100       ┆ 54         ┆ 86         ┆ 14        ┆ 12        ┆ 0.54      ┆ 0.


## Drop customers that are still incomplete

After imputation, a customer either has a full year of data (8,758 hourly rows, real or imputed) or has gaps too long to fill reliably: they joined late, left early, or have gaps longer than three days. We **drop the second group from the analysis**, so every remaining customer covers the same full year. That way averages, seasonal patterns and group comparisons aren't skewed by customers who are missing whole seasons.

This is done in DuckDB, without deleting anything:

- **`excluded_customers`** (table): one row per dropped customer, with how many rows they have, their first and last dates, and a **reason**:
  - *joined late*: no data at the start of the year;
  - *left early*: no data at the end of the year;
  - *long gap mid-year*: a gap of more than three days in the middle of the year.

  A customer who both joined late and left early is labelled *joined late*. The table keeps the decision visible and easy to revisit.
- **`ami_analysis`** (view): `ami_filled` (real plus imputed rows) **minus** the excluded customers. **This is the dataset to use for analysis from here on.** Because it's a view, it uses no extra disk space and always reflects the current `ami`, `ami_imputed_rows` and `excluded_customers` tables.

The raw `ami` table and `ami_filled` stay unchanged, so the excluded customers can still be looked at if needed.

EV, HP and PV customers with no meter data at all aren't in `ami`, so there's nothing to drop for them. They are simply absent from `ami_analysis`.


In [12]:

DATA_START, DATA_END = '2025-09-01', '2026-08-31'

DROP_SQL = f"""
CREATE OR REPLACE TABLE excluded_customers AS
SELECT CUSTOMER_NUMBER,
       COUNT(*) AS row_count_filled,
       {EXPECTED_ROWS} - COUNT(*) AS missing_hours,
       MIN(usage_date) AS first_date,
       MAX(usage_date) AS last_date,
       CASE WHEN MIN(usage_date) > DATE '{DATA_START}' THEN 'joined late'
            WHEN MAX(usage_date) < DATE '{DATA_END}' THEN 'left early'
            ELSE 'long gap mid-year'
       END AS reason
FROM ami_filled
GROUP BY CUSTOMER_NUMBER
HAVING COUNT(*) <> {EXPECTED_ROWS}
ORDER BY CUSTOMER_NUMBER;

CREATE OR REPLACE VIEW ami_analysis AS
SELECT * FROM ami_filled
ANTI JOIN excluded_customers USING (CUSTOMER_NUMBER);
"""

with duckdb.connect(DB_PATH) as con:
    con.execute(DROP_SQL)
    excluded_by_reason = con.sql("""
        SELECT reason, COUNT(*) AS customers, SUM(missing_hours) AS missing_hours,
               MEDIAN(missing_hours) AS median_missing_hours
        FROM excluded_customers GROUP BY 1 ORDER BY 2 DESC
    """).pl()
    analysis_check = con.sql(f"""
        SELECT COUNT(DISTINCT CUSTOMER_NUMBER) AS customers,
               COUNT(*) AS rows,
               COUNT(*) FILTER (WHERE is_imputed) AS imputed_rows,
               COUNT(DISTINCT CUSTOMER_NUMBER) * {EXPECTED_ROWS} AS expected_rows
        FROM ami_analysis
    """).pl()
    analysis_customers = con.sql(
        'SELECT DISTINCT CUSTOMER_NUMBER FROM ami_analysis'
    ).pl()

# Every remaining customer should have exactly one full year of hourly rows.
assert analysis_check['rows'].item() == analysis_check['expected_rows'].item()

with pl.Config(tbl_cols=-1, tbl_hide_dataframe_shape=True, thousands_separator=True):
    print('Excluded customers by reason:')
    print(excluded_by_reason)
    print('\nami_analysis:')
    print(analysis_check)


Excluded customers by reason:
┌───────────────────┬───────────┬───────────────┬──────────────────────┐
│ reason            ┆ customers ┆ missing_hours ┆ median_missing_hours │
│ ---               ┆ ---       ┆ ---           ┆ ---                  │
│ str               ┆ i64       ┆ decimal[38,0] ┆ f64                  │
╞═══════════════════╪═══════════╪═══════════════╪══════════════════════╡
│ joined late       ┆ 369       ┆ 664,705       ┆ 1,427.0              │
│ left early        ┆ 172       ┆ 647,916       ┆ 3,924.0              │
│ long gap mid-year ┆ 131       ┆ 232,204       ┆ 1,344.0              │
└───────────────────┴───────────┴───────────────┴──────────────────────┘

ami_analysis:
┌───────────┬─────────────┬──────────────┬───────────────┐
│ customers ┆ rows        ┆ imputed_rows ┆ expected_rows │
│ ---       ┆ ---         ┆ ---          ┆ ---           │
│ i64       ┆ i64         ┆ i64          ┆ i64           │
╞═══════════╪═════════════╪══════════════╪═══════════════╡
│ 3


### EV, HP and PV customers in the analysis dataset

This is the final count of each group's customers in `ami_analysis`, the customers the rest of the analysis will use. It is taken directly from the view, so it confirms the drop worked.


In [13]:

in_analysis = analysis_customers.with_columns(pl.lit(True).alias('in_analysis'))

final_counts = pl.concat([
    customers.join(in_analysis, on='CUSTOMER_NUMBER', how='left')
    .select(
        pl.lit(tab).alias('tab'),
        pl.len().alias('customers'),
        pl.col('in_analysis').fill_null(False).sum().alias('in_analysis'),
        (~pl.col('in_analysis').fill_null(False)).sum().alias('dropped_or_no_data'),
    )
    for tab, customers in groups.items()
]).with_columns(
    (pl.col('in_analysis') / pl.col('customers')).round(3).alias('share_in_analysis')
)

print(f'All customers: {analysis_customers.height:,} of {row_counts.height:,} '
      f'({analysis_customers.height / row_counts.height:.1%}) are in ami_analysis.')
final_counts


All customers: 33,566 of 34,238 (98.0%) are in ami_analysis.


tab,customers,in_analysis,dropped_or_no_data,share_in_analysis
str,u32,u32,u32,f64
"""EV Customers""",200,179,21,0.895
"""HP Customers""",100,86,14,0.86
"""PV Customers""",434,389,45,0.896


## Tried but not used

| Tried | Why it isn't used |
|---|---|
| **Dropping every incomplete customer** (keeping only customers with a complete year) | Keeps only about three-quarters of all customers and just over half of the HP list (the cut-off table above). Most gaps are one- to three-day outages that hit thousands of meters at once, not faulty meters, so those customers are imputed instead. |
| **Keeping customers with longer gaps** (up to 7 or 30 days missing) | Would add only 0.2–0.5% more customers than the 72-hour rule (98.2% kept at 7 days and 98.5% at 30, against about 98% now), and filling weeks of data would mostly be invention. Those customers are dropped (`excluded_customers`). |
| **Leaving out the 10,001–20,000 folder** | The first delivery of that folder duplicated data from the other folders, so it was disregarded. The corrected folder (received 13 Nov 2026) is now loaded like the others. It is still missing 7 of its 51 part files (parts 41 and 44–49), so about 1,400 customer numbers have no meter data yet. |